In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 28. Data Parallelism

> **Learning Goals**
> - Understand the communication pattern of data parallelism (All-Reduce)
> - Explain the mathematics of gradient synchronization
> - Try DDP (DistributedDataParallel)

## 28.1 Limits of a Single GPU

LLM training has enormous memory and compute requirements:
- 7B model in FP16 + AdamW is about 70GB
- Larger batches require more memory
- A single GPU can take months

Solution: distribute training across multiple GPUs.

## 28.2 Basic Idea of Data Parallelism

1. Place $N$ copies of the same model on $N$ GPUs
2. Split data into $N$ shards and assign one shard to each GPU
3. Each GPU computes gradients on its own data
4. Average the gradients and synchronize them across all GPUs
5. Each GPU applies the same update, keeping the model consistent

Mathematically:
- Local gradient: $\mathbf{g}_i = \nabla \mathcal{L}_i(\theta)$
- Synchronization: $\mathbf{g} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{g}_i$
- Update: $\theta \leftarrow \theta - \eta \mathbf{g}$

Effective batch size: $B_{\text{eff}} = N \cdot B_{\text{local}}$


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Data-parallel simulation on a single GPU using gradient accumulation
class DataParallelSimulator:
    """simulator that mimics N-GPU DDP on one GPU."""
    def __init__(self, model, n_gpus=4):
        self.model = model
        self.n_gpus = n_gpus
        # N synthetic GPUs = copies of the same model simulated with gradient accumulation
        # in practice this accumulates gradients n_gpus times in one model
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    def step(self, x_batch, y_batch):
        """x_batch: (N*B, ...), N GPU B samples each."""
        # 1. split the batch into N parts
        chunks = torch.chunk(x_batch, self.n_gpus)
        y_chunks = torch.chunk(y_batch, self.n_gpus)

        # 2. compute gradients on each synthetic GPU (gradient accumulation)
        self.optimizer.zero_grad()
        total_loss = 0
        for i in range(self.n_gpus):
            out = self.model(chunks[i])
            loss = F.mse_loss(out, y_chunks[i]) / self.n_gpus  # Mean
            loss.backward()  # Gradient cumulative sum
            total_loss += loss.item() * self.n_gpus

        # 3. optimizer.step() uses accumulated gradients, already averaged by scaling the loss
        self.optimizer.step()
        return total_loss / self.n_gpus

# simulate with a small model
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 20)
)

dp = DataParallelSimulator(model, n_gpus=4)

# Comparison: single batch vs DDP simulation
x = torch.randn(64, 20)
y = torch.randn(64, 20)

loss = dp.step(x, y)
print(f"DDP simulation 1 step loss: {loss:.4f}")
print(f"n_gpus=4, batch=64 → per-GPU batch=16")
print(f"effective batch size: {dp.n_gpus * 16} = 64")


## 28.3 All-Reduce Communication

Each GPU propagates its gradients to all other GPUs:
- **Ring All-Reduce**: $2(N-1)$ steps; each step sends $|\theta|/N$ data
- **Tree All-Reduce**: logarithmic steps

Communication volume is $O(|\theta|)$ per step. A 7B model sends about 14GB per step.

This communication can be a bottleneck, so systems hide it with communication-computation overlap.


In [ ]:
# All-Reduce simulation
def all_reduce_sum(grads_list):
    """average gradients from each GPU and distribute them to all GPUs."""
    # average
    total = sum(grads_list)
    # assign the same values to all GPUs
    return [total.clone() for _ in grads_list]

# 4 GPU simulation
torch.manual_seed(0)
n_gpus = 4
local_grads = [torch.randn(5) for _ in range(n_gpus)]
print("local gradients on each GPU:")
for i, g in enumerate(local_grads):
    print(f"  GPU {i}: {g.numpy().round(3)}")

# All-Reduce (Mean)
reduced = all_reduce_sum(local_grads)
avg = [r / n_gpus for r in reduced]
print("\nAll-Reduce after (mean):")
for i, g in enumerate(avg):
    print(f"  GPU {i}: {g.numpy().round(3)}")
print("\n=> all GPUs have the same mean gradient.")


## 28.4 PyTorch DDP (Concept)

Actual PyTorch code:
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group('nccl', rank=rank, world_size=world_size)
model = model.to(rank)
model = DDP(model, device_ids=[rank])
# model(x) is distributed automatically after this
```

Colab with a single GPU cannot run true multi-GPU DDP, but gradient accumulation can simulate a similar effect.


In [ ]:
# single-GPU DDP simulation
import time
from llm_math.bench import time_fn

# DDP vs single batch for a larger model
def make_model(d=512):
    return nn.Sequential(
        nn.Linear(d, d * 2), nn.ReLU(),
        nn.Linear(d * 2, d), nn.ReLU(),
        nn.Linear(d, d)
    )

# single large batch
torch.manual_seed(0)
model_single = make_model()
opt_single = torch.optim.AdamW(model_single.parameters(), lr=1e-3)

def step_single(x, y):
    opt_single.zero_grad()
    out = model_single(x)
    loss = F.mse_loss(out, y)
    loss.backward()
    opt_single.step()
    return loss

# DDP simulation: 4 GPUs, each with one quarter of the batch
torch.manual_seed(0)
model_ddp = make_model()
opt_ddp = torch.optim.AdamW(model_ddp.parameters(), lr=1e-3)

def step_ddp(x, y, n_gpus=4):
    opt_ddp.zero_grad()
    x_chunks = torch.chunk(x, n_gpus)
    y_chunks = torch.chunk(y, n_gpus)
    total_loss = 0
    for xc, yc in zip(x_chunks, y_chunks):
        out = model_ddp(xc)
        loss = F.mse_loss(out, yc) / n_gpus  # mean
        loss.backward()  # gradient cumulative sum
        total_loss += loss.item() * n_gpus
    opt_ddp.step()
    return total_loss / n_gpus

# Time comparison with the same total batch size
d = 512
batch_size = 128
x = torch.randn(batch_size, d)
y = torch.randn(batch_size, d)

t_single = time_fn(step_single, x, y, device='cpu', warmup=2, repeat=5)
t_ddp = time_fn(step_ddp, x, y, device='cpu', warmup=2, repeat=5)
print(f"single large batch (B=128): {t_single['mean_ms']:.3f} ms")
print(f"DDP simulation (4 GPU, each B=32): {t_ddp['mean_ms']:.3f} ms")
print("\n=> DDP simulation is slower on a single GPU because it runs sequentially.")
print("   real multi-GPU parallelism is much faster.")


## 28.5 Large Batch Training — LARS, LAMB

Large batches can destabilize training. Layer-wise learning-rate scaling helps.

**LARS** (Layer-wise Adaptive Rate Scaling):
$$\eta_\ell = \eta \cdot \frac{\|\theta_\ell\|}{\|\nabla \mathcal{L}_\ell\| + \lambda \|\theta_\ell\|}$$

**LAMB**: LARS + Adam. Used in large-scale LLM training.

## 28.6 Communication-Computation Overlap

Backpropagation proceeds layer by layer. As soon as a layer's gradient is computed, All-Reduce starts while the next layer's backward pass continues.

This can hide most communication time.

## 28.7 Key Takeaways

| Concept | Description |
|---|---|
| Data parallelism | N copies of the same model, split data |
| All-Reduce | gradient synchronization |
| Effective batch | $N \cdot B_{\text{local}}$ |
| Communication volume | $O(\|\theta\|)$ per step |
| LARS/LAMB | stabilize large-batch training |

## Exercises

1. In a 4-GPU DDP simulation, train each GPU on different data and show that synchronized gradients match.
2. Train with effective batch sizes 32, 64, 128, and 256, then compare convergence.
3. Explain why All-Reduce communication is $O(\|\theta\|)$.
4. Explain why communication-computation overlap improves speed.
5. Explain why LARS helps with large batches.

> Solutions: `solutions/ch28_solutions.ipynb`
